# 02 · Subconjunto RSNA Knee (Fase 3)

Entradas (Add Input):
1. La competencia `rsna-knee-abnormality-detection`.
2. El Dataset privado `rsna-knee-weak-labels-v1` (`weak_labels.csv`, `gold_split.csv`).
3. El output del notebook `01_eda_rsna_knee` (`series_catalog.csv`, `study_lang.csv`).

Salida en `/kaggle/working`: `vol/*.npy` (uint8, cortes × 256 × 256), `subset_studies.csv`,
`subset_manifest.csv`, `prevalence_*.csv` y `conversion_errors.csv`. Se publica como Dataset privado `rsna-knee-subset-v1`.

Política de etiquetas: solo el estado POS cuenta como 1. MILD, MIN, NEG y NONE valen 0. UNC y WEAK quedan en 0 con máscara 0.
**No usar la columna `<label>` de weak_labels.csv**, porque ahí MILD figura como 1.

In [ ]:
import glob, os, time
from pathlib import Path
import numpy as np
import pandas as pd

def find(pattern, max_depth=4):
    # Sin glob recursivo: bajar por train_series/ recorrería los 819,640 DICOM.
    for d in range(1, max_depth + 1):
        hits = sorted(glob.glob("/kaggle/input/" + "*/" * (d - 1) + pattern))
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(f"no encuentro {pattern} en /kaggle/input (¿falta Add Input?)")

COMP = find("*knee*abnormal*")
assert (COMP / "train.csv").exists(), COMP
WEAK = find("weak_labels.csv")
GOLD_SPLIT = find("gold_split.csv")
CATALOG = find("series_catalog.csv")
STUDY_LANG = find("study_lang.csv")
WORK = Path("/kaggle/working"); VOL = WORK / "vol"; VOL.mkdir(parents=True, exist_ok=True)
for p in [COMP, WEAK, GOLD_SPLIT, CATALOG, STUDY_LANG]:
    print(p)

N_TRAIN_POOL, MIN_POS, VAL_FRAC, SEED = 500, 60, 0.2, 42

In [ ]:
# ===== src/data/select_subset.py (copiado tal cual del repo) =====
"""Fase 3: política de etiquetas, selección de series, muestreo y split del subconjunto.

Solo usa pandas/numpy, así que se prueba en el Mac sin imágenes. El notebook
`02_subset_rsna_knee` incluye este archivo tal cual.
"""
import numpy as np
import pandas as pd

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
    "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]

# Política fijada en la Fase 2 antes de abrir test: solo POS es 1.
# OJO: en la columna `<label>` de weak_labels.csv MILD figura como 1; por eso se usa `__state`.
STATE_TO_Y = {"POS": 1, "MILD": 0, "MIN": 0, "NEG": 0, "NONE": 0, "UNC": -1, "WEAK": -1}

# Roles de serie: (plano, combinación preferida, respaldo) con combinación = (fluid, FS).
ROLES = {"sag": ("Sagittal", (1, 1), (0, 0)), "cor": ("Coronal", (1, 1), (0, 0))}
THIN_MM = 1.5      # por debajo se considera adquisición 3D
TARGET_SLICES = 30


def apply_policy(weak):
    """weak_labels.csv → una fila por estudio con y_, mask_, conf_, score_ y state_ por etiqueta.

    y_ vale 1/0 (los -1 quedan como 0) y mask_ vale 0 donde el estado es UNC o WEAK.
    """
    out = {"StudyInstanceUID": weak["StudyInstanceUID"].values}
    for lab in LABELS:
        state = weak[f"{lab}__state"]
        unknown = set(state.unique()) - set(STATE_TO_Y)
        if unknown:
            raise ValueError(f"{lab}: estados desconocidos {unknown}")
        y = state.map(STATE_TO_Y)
        out[f"y_{lab}"] = y.clip(lower=0).astype(int).values
        out[f"mask_{lab}"] = (y != -1).astype(int).values
        out[f"conf_{lab}"] = weak[f"{lab}__conf"].values
        out[f"score_{lab}"] = weak[f"{lab}__score"].values
        out[f"state_{lab}"] = state.values
    return pd.DataFrame(out)


def _pick(cands):
    """Entre varias series candidatas: 2D primero, luego cortes cerca de 30, luego mayor matriz."""
    c = cands.assign(
        _is2d=(cands["SliceThickness"].fillna(0) >= 2).astype(int),
        _dist=(cands["n_slices"] - TARGET_SLICES).abs(),
    )
    c = c.sort_values(["_is2d", "_dist", "Rows"], ascending=[False, True, False])
    return c.iloc[0]


def choose_series(catalog):
    """series_catalog.csv → una fila por estudio con la serie elegida para cada rol.

    Columnas por rol: <rol>_series, <rol>_fallback (1 si se usó fluid=0 FS=0),
    <rol>_n_slices, <rol>_thick, <rol>_is3d.
    """
    rows = []
    for uid, g in catalog.groupby("StudyInstanceUID", sort=False):
        r = {"StudyInstanceUID": uid}
        for role, (plane, pref, back) in ROLES.items():
            pick, fallback = None, 0
            for i, (fl, fs) in enumerate((pref, back)):
                m = g[(g["Anatomical_Plane"] == plane) & (g["Fluid_Sensitive"] == fl)
                      & (g["Fat_Suppression"] == fs) & (g["n_slices"] > 0)]
                if len(m):
                    pick, fallback = _pick(m), i
                    break
            if pick is None:
                r.update({f"{role}_series": None, f"{role}_fallback": np.nan,
                          f"{role}_n_slices": np.nan, f"{role}_thick": np.nan, f"{role}_is3d": np.nan})
            else:
                th = pick["SliceThickness"]
                r.update({f"{role}_series": pick["SeriesInstanceUID"], f"{role}_fallback": fallback,
                          f"{role}_n_slices": int(pick["n_slices"]), f"{role}_thick": th,
                          f"{role}_is3d": int(pd.notna(th) and th < THIN_MM)})
        g0 = g.iloc[0]
        r["Manufacturer"] = g0.get("Manufacturer")
        r["FieldStrength"] = g0.get("FieldStrength")
        rows.append(r)
    return pd.DataFrame(rows)


def eligible_for_training(series):
    """Estudios no dorados utilizables: sagital y coronal presentes y ninguna 3D."""
    s = series
    ok = s["sag_series"].notna() & s["cor_series"].notna() & (s["sag_is3d"] == 0) & (s["cor_is3d"] == 0)
    return set(s.loc[ok, "StudyInstanceUID"])


def sample_studies(labels, eligible, n=500, min_pos=60, seed=42):
    """Muestreo estratificado: primero asegura `min_pos` positivos (POS) por etiqueta,
    empezando por la más rara, y completa hasta `n` al azar.

    Devuelve la lista de UIDs elegidos (orden reproducible).
    """
    rng = np.random.default_rng(seed)
    pool = labels[labels["StudyInstanceUID"].isin(eligible)].set_index("StudyInstanceUID")
    pool = pool.loc[rng.permutation(pool.index.values)]
    chosen = []
    chosen_set = set()
    order = sorted(LABELS, key=lambda lab: pool[f"y_{lab}"].sum())
    for lab in order:
        have = int(pool.loc[chosen, f"y_{lab}"].sum()) if chosen else 0
        if have >= min_pos:
            continue
        cands = [u for u in pool.index[pool[f"y_{lab}"] == 1] if u not in chosen_set]
        for u in cands[: min_pos - have]:
            chosen.append(u)
            chosen_set.add(u)
    if len(chosen) > n:
        raise ValueError(f"min_pos={min_pos} exige {len(chosen)} estudios > n={n}")
    rest = [u for u in pool.index if u not in chosen_set]
    chosen += rest[: n - len(chosen)]
    return chosen


def split_train_val(labels, uids, val_frac=0.2, seed=42):
    """Split por estudio. Reparte los positivos de cada etiqueta (de la más rara a la más
    común) para que val tenga cerca de `val_frac` de ellos. Devuelve {uid: 'train'|'val'}."""
    rng = np.random.default_rng(seed)
    df = labels.set_index("StudyInstanceUID").loc[list(uids)]
    df = df.loc[rng.permutation(df.index.values)]
    n_val = int(round(val_frac * len(df)))
    split = {}
    order = sorted(LABELS, key=lambda lab: df[f"y_{lab}"].sum())
    for lab in order:
        pos = df.index[df[f"y_{lab}"] == 1]
        target = int(round(val_frac * len(pos)))
        have = sum(split.get(u) == "val" for u in pos)
        for u in pos:
            if u in split:
                continue
            if have < target and sum(v == "val" for v in split.values()) < n_val:
                split[u] = "val"
                have += 1
            else:
                split[u] = "train"
    n_now = sum(v == "val" for v in split.values())
    for u in df.index:
        if u not in split:
            split[u] = "val" if n_now < n_val else "train"
            n_now += split[u] == "val"
    return split


def prevalence(labels, uids, by=None):
    """Positivos y % por etiqueta (y_ == 1) sobre los estudios dados; opcionalmente por grupo."""
    df = labels[labels["StudyInstanceUID"].isin(set(uids))]
    def one(d):
        return pd.Series({**{lab: int(d[f"y_{lab}"].sum()) for lab in LABELS}, "n": len(d)})
    if by is None:
        t = one(df).to_frame("pos")
        t["pct"] = (t["pos"] / len(df) * 100).round(1)
        t.loc["n", "pct"] = np.nan
        return t
    return df.groupby(by).apply(one)

In [ ]:
# ===== src/data/dicom_to_npy.py (copiado tal cual del repo) =====
"""Fase 3: serie DICOM → volumen uint8 (cortes, 256, 256) guardado como .npy.

Pasos: ordenar cortes por InstanceNumber (respaldo: posición sobre la normal),
normalizar por percentiles 1 a 99 del volumen, rellenar a cuadrado y redimensionar.
Las series con más de MAX_SLICES cortes (3D finas) se reagrupan en TARGET_SLICES
losas promediando cortes contiguos.
"""
from pathlib import Path

import cv2
import numpy as np
import pydicom

SIZE = 256
MAX_SLICES = 60
TARGET_SLICES = 30


def _sort_key(ds):
    inst = getattr(ds, "InstanceNumber", None)
    pos = getattr(ds, "ImagePositionPatient", None)
    ori = getattr(ds, "ImageOrientationPatient", None)
    if pos is not None and ori is not None:
        normal = np.cross(np.array(ori[:3], float), np.array(ori[3:], float))
        proj = float(np.dot(normal, np.array(pos, float)))
    else:
        proj = 0.0
    return (int(inst) if inst is not None else 0, proj)


def read_series(series_dir):
    """Lee todos los .dcm de una carpeta y devuelve (volumen float32, metadatos)."""
    files = sorted(Path(series_dir).glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"sin .dcm en {series_dir}")
    dss = [pydicom.dcmread(f) for f in files]
    dss.sort(key=_sort_key)
    slices = []
    for ds in dss:
        a = ds.pixel_array.astype(np.float32)
        if a.ndim != 2:  # multiframe u otro formato raro: se omite el corte
            continue
        a = a * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
        if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
            a = a.max() - a
        slices.append(a)
    ds0 = dss[0]
    meta = {
        "rows": int(ds0.Rows), "cols": int(ds0.Columns),
        "pixel_spacing": float(ds0.PixelSpacing[0]) if "PixelSpacing" in ds0 else np.nan,
        "n_files": len(files),
    }
    return slices, meta


def normalize(slices, lo=1, hi=99):
    """Percentiles 1 a 99 calculados sobre todo el volumen → [0, 1]."""
    allpx = np.concatenate([s.ravel()[:: max(1, s.size // 20000)] for s in slices])
    p_lo, p_hi = np.percentile(allpx, [lo, hi])
    scale = max(p_hi - p_lo, 1e-6)
    return [np.clip((s - p_lo) / scale, 0, 1) for s in slices]


def to_square(img, size=SIZE):
    """Rellena con ceros hasta cuadrado (centrado) y redimensiona sin deformar."""
    h, w = img.shape
    m = max(h, w)
    out = np.zeros((m, m), img.dtype)
    y0, x0 = (m - h) // 2, (m - w) // 2
    out[y0:y0 + h, x0:x0 + w] = img
    interp = cv2.INTER_AREA if m > size else cv2.INTER_LINEAR
    return cv2.resize(out, (size, size), interpolation=interp)


def regroup(vol, target=TARGET_SLICES):
    """(n, H, W) con n grande → (target, H, W) promediando grupos contiguos (losas)."""
    edges = np.linspace(0, len(vol), target + 1).round().astype(int)
    return np.stack([vol[a:b].mean(0) for a, b in zip(edges[:-1], edges[1:])])


def convert_series(series_dir, out_path, size=SIZE):
    """Convierte una serie y guarda uint8 (n, size, size). Devuelve metadatos para el manifiesto."""
    slices, meta = read_series(series_dir)
    slices = normalize(slices)
    vol = np.stack([to_square(s, size) for s in slices]).astype(np.float32)
    regrouped = len(vol) > MAX_SLICES
    if regrouped:
        vol = regroup(vol)
    vol = (vol * 255).round().astype(np.uint8)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    np.save(out_path, vol)
    meta.update({"n_slices_out": int(vol.shape[0]), "regrouped": int(regrouped),
                 "bytes": int(vol.nbytes)})
    return meta

## 1. Etiquetas con la política (desde `__state`)

In [ ]:
weak = pd.read_csv(WEAK)
gold_split = pd.read_csv(GOLD_SPLIT)
train = pd.read_csv(COMP / "train.csv")
lang = pd.read_csv(STUDY_LANG, usecols=["StudyInstanceUID", "lang"])
labels = apply_policy(weak)
GOLD = set(gold_split["StudyInstanceUID"])
assert len(weak) == 4407 and len(GOLD) == 58

# Comprobación de la advertencia: la columna cruda cuenta MILD como 1 y la política no.
chk = pd.DataFrame({lab: [int((weak[lab] == 1).sum()), int(labels[f"y_{lab}"].sum()),
                          int((labels[f"mask_{lab}"] == 0).sum())] for lab in LABELS},
                   index=["col_cruda==1", "y_politica", "enmascarados"]).T
print(chk)
assert labels["y_Effusion"].sum() == (weak["Effusion__state"] == "POS").sum()

## 2. Selección de series por estudio

In [ ]:
catalog = pd.read_csv(CATALOG)
t0 = time.time()
series = choose_series(catalog)
print(f"{len(series)} estudios en {time.time()-t0:.0f} s")
for role in ROLES:
    s = series[f"{role}_series"].notna()
    print(f"{role}: presente {s.sum()} · respaldo 0/0 {int((series[f'{role}_fallback'] == 1).sum())}"
          f" · 3D {int((series[f'{role}_is3d'] == 1).sum())}")
g = series[series.StudyInstanceUID.isin(GOLD)]
print("dorados sin sag:", g.sag_series.isna().sum(), "· sin cor:", g.cor_series.isna().sum(),
      "· con respaldo sag/cor:", int(g.sag_fallback.sum()), int(g.cor_fallback.sum()),
      "· 3D sag/cor:", int(g.sag_is3d.sum()), int(g.cor_is3d.sum()))
eligible = eligible_for_training(series) - GOLD
print("no dorados elegibles (sag y cor 2D):", len(eligible), "de", len(series) - len(GOLD))

## 3. Muestreo estratificado, split y prevalencia (entregable 6)

In [ ]:
pool_uids = sample_studies(labels, eligible, n=N_TRAIN_POOL, min_pos=MIN_POS, seed=SEED)
split = split_train_val(labels, pool_uids, val_frac=VAL_FRAC, seed=SEED)
split.update({u: "test" for u in GOLD})
uids = list(pool_uids) + sorted(GOLD)

studies = (pd.DataFrame({"StudyInstanceUID": uids})
           .assign(split=lambda d: d.StudyInstanceUID.map(split), is_gold=lambda d: d.StudyInstanceUID.isin(GOLD).astype(int))
           .merge(gold_split.rename(columns={"split": "gold_split"})[["StudyInstanceUID", "gold_split"]], how="left")
           .merge(lang, how="left")
           .merge(series, how="left")
           .merge(labels, how="left")
           .merge(train[["StudyInstanceUID"] + LABELS].rename(columns={l: f"gold_{l}" for l in LABELS}), how="left"))
studies.loc[studies.is_gold == 0, [f"gold_{l}" for l in LABELS]] = np.nan
assert studies.StudyInstanceUID.is_unique and studies[[f"gold_{l}" for l in LABELS]].iloc[-58:].notna().all().all()
print(studies.split.value_counts())

all_nongold = set(labels.StudyInstanceUID) - GOLD
prev = pd.concat({
    "pool_4349": prevalence(labels, all_nongold)["pct"],
    "subset_500": prevalence(labels, pool_uids)["pct"],
    "train_pos": prevalence(labels, studies.query("split=='train'").StudyInstanceUID)["pos"],
    "val_pos": prevalence(labels, studies.query("split=='val'").StudyInstanceUID)["pos"],
    "gold58_pos": pd.Series({**{l: int(train.loc[train.StudyInstanceUID.isin(GOLD), l].sum()) for l in LABELS}, "n": 58}),
}, axis=1)
prev.to_csv(WORK / "prevalence_subset.csv")
print(prev)
by_lang = pd.crosstab(studies.lang, studies.split)
by_lang.to_csv(WORK / "subset_by_lang.csv"); print(by_lang)
by_man = pd.crosstab(studies.Manufacturer, studies.split)
by_man.to_csv(WORK / "subset_by_manufacturer.csv"); print(by_man)

## 4. Prueba de conversión y códecs (antes del bucle largo)

In [ ]:
import pydicom
jobs = []
for r in studies.itertuples():
    for role in ROLES:
        sid = getattr(r, f"{role}_series")
        if isinstance(sid, str):
            jobs.append((r.StudyInstanceUID, role, sid))
print(len(jobs), "series a convertir")

# Sintaxis de transferencia (solo cabecera del primer archivo de cada serie)
ts = {}
for st, role, sid in jobs:
    f = next((COMP / "train_series" / st / sid).glob("*.dcm"))
    t = str(pydicom.dcmread(f, stop_before_pixels=True).file_meta.TransferSyntaxUID.name)
    ts[t] = ts.get(t, 0) + 1
print(ts)
if any(("JPEG" in k) or ("RLE" in k) for k in ts):
    print("Hay series comprimidas: si la prueba de abajo falla, correr  !pip install -q pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg  (requiere Internet en Settings)")

t0 = time.time()
for st, role, sid in jobs[:3]:
    print(convert_series(COMP / "train_series" / st / sid, VOL / f"{st}_{role}.npy"))
print(f"{(time.time()-t0)/3:.1f} s por serie → estimado {len(jobs)*(time.time()-t0)/3/60/max(os.cpu_count(),1):.0f} min con {os.cpu_count()} procesos")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, (st, role, sid) in zip(ax, jobs[:3]):
    v = np.load(VOL / f"{st}_{role}.npy"); a.imshow(v[len(v)//2], cmap="gray"); a.set_title(f"{role} {v.shape}"); a.axis("off")
plt.show()

## 5. Conversión completa (reanudable: salta los .npy ya guardados)

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

def run(job):
    st, role, sid = job
    out = VOL / f"{st}_{role}.npy"
    try:
        if out.exists():
            v = np.load(out, mmap_mode="r")
            return {**dict(zip(["StudyInstanceUID", "role", "SeriesInstanceUID"], job)), "n_slices_out": v.shape[0], "skipped": 1}
        meta = convert_series(COMP / "train_series" / st / sid, out)
        return {"StudyInstanceUID": st, "role": role, "SeriesInstanceUID": sid, **meta}
    except Exception as e:
        return {"StudyInstanceUID": st, "role": role, "SeriesInstanceUID": sid, "error": repr(e)[:300]}

t0, results = time.time(), []
with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mp.get_context("fork")) as ex:
    futs = [ex.submit(run, j) for j in jobs]
    for i, f in enumerate(as_completed(futs), 1):
        results.append(f.result())
        if i % 100 == 0 or i == len(jobs):
            print(f"{i}/{len(jobs)} · {(time.time()-t0)/60:.1f} min", flush=True)
res = pd.DataFrame(results)
errs = res[res.get("error").notna()] if "error" in res else res.iloc[:0]
errs.to_csv(WORK / "conversion_errors.csv", index=False)
print("errores:", len(errs)); print(errs.head())

## 6. Manifiesto y resumen

In [ ]:
plane = {"sag": "Sagittal", "cor": "Coronal"}
ok = res[res.get("error").isna()] if "error" in res else res
man = ok.assign(plane=ok.role.map(plane), path=lambda d: "vol/" + d.StudyInstanceUID + "_" + d.role + ".npy")
man["n_slices"] = [np.load(WORK / p, mmap_mode="r").shape[0] for p in man.path]
man["fallback"] = [studies.set_index("StudyInstanceUID").at[s, f"{r}_fallback"] for s, r in zip(man.StudyInstanceUID, man.role)]
keep = ["StudyInstanceUID", "split", "is_gold", "gold_split", "lang", "Manufacturer", "FieldStrength"] \
       + [f"{p}_{l}" for l in LABELS for p in ("y", "mask", "conf", "score")] + [f"gold_{l}" for l in LABELS]
man = man[["StudyInstanceUID", "role", "plane", "SeriesInstanceUID", "path", "n_slices", "fallback"]].merge(studies[keep], how="left")
man = man.sort_values(["split", "StudyInstanceUID", "role"])
man.to_csv(WORK / "subset_manifest.csv", index=False)

# Estudios con su estado de conversión; un estudio sin sagital válida no sirve para entrenar
have = man.pivot_table(index="StudyInstanceUID", columns="role", values="n_slices", aggfunc="first")
studies["has_sag"] = studies.StudyInstanceUID.map(have.get("sag")).notna().astype(int)
studies["has_cor"] = studies.StudyInstanceUID.map(have.get("cor")).notna().astype(int)
studies.to_csv(WORK / "subset_studies.csv", index=False)
print(studies.groupby("split")[["has_sag", "has_cor"]].sum())
size_gb = sum(f.stat().st_size for f in VOL.glob("*.npy")) / 1e9
print(f"{len(man)} series · {size_gb:.2f} GB · cortes por serie: {man.n_slices.describe()[['min','50%','max']].to_dict()}")

In [ ]:
# Control visual: 4 estudios al azar de cada split, corte central sagital y coronal
fig, ax = plt.subplots(3, 8, figsize=(20, 8))
rng = np.random.default_rng(0)
for i, sp in enumerate(["train", "val", "test"]):
    u = rng.choice(studies.query("split==@sp").StudyInstanceUID.values, 4, replace=False)
    for j, st in enumerate(u):
        for k, role in enumerate(ROLES):
            a = ax[i, 2*j + k]; p = VOL / f"{st}_{role}.npy"
            if p.exists():
                v = np.load(p); a.imshow(v[len(v)//2], cmap="gray")
            a.set_title(f"{sp} {role}", fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

## 7. Publicar

1. **Save Version → Save & Run All (Commit)**. La conversión corre en el commit, así que la sesión interactiva se puede cerrar.
2. En la versión guardada: **Output → New Dataset** → nombre `rsna-knee-subset-v1` → **Private**.
3. Cuando exista el Team, compartir el Dataset con los integrantes (Settings → Sharing). Nunca hacerlo público ni subirlo a GitHub.